# 🌊 Smoke Run WAVE-CRE (Baseline B5 Comparator)
### Adaptive Prompting for Continual Relation Extraction (FewRel 10 tasks × 8 relations, Seed 2021)

Notebook này phục vụ việc thực thi kiểm chứng (smoke run) 1 seed duy nhất cho baseline **B5 (WAVE-CRE)** trên môi trường GPU của Google Colab.
- **Repository:** `research-code-lab` (Branch: `feat/b5-wave-smoke`)
- **Mã nguồn:** `paper-reproductions/WAVE-CRE` (Upstream commit: `5273b70`)
- **Mục tiêu:** Chạy toàn bộ 10 task với config chuẩn của tác giả, xuất ma trận `metrics.json` theo Protocol v1 và tự động sao lưu artifact lên Google Drive.

## 1. Kiểm tra Môi trường GPU (NVIDIA CUDA Verification)

In [ ]:
import torch
import os
import sys

print("=" * 60)
print("CUDA / GPU PREFLIGHT CHECK")
print("=" * 60)
if torch.cuda.is_available():
    print(f"✓ CUDA Available: True")
    print(f"✓ GPU Model:     {torch.cuda.get_device_name(0)}")
    print(f"✓ VRAM:          {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    !nvidia-smi
else:
    print("⚠️ CẢNH BÁO: CUDA không khả dụng! Hãy vào Runtime -> Change runtime type -> Chọn T4 GPU.")

## 2. (Tùy chọn Khuyến nghị) Kết nối Google Drive để Bảo Toàn Artifacts
Chạy cell này để mount Drive cá nhân, đảm bảo sau khi huấn luyện xong mọi kết quả và log không bị mất nếu Colab ngắt kết nối.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

GDRIVE_BACKUP_DIR = Path('/content/drive/MyDrive/research-code-lab-runs/wave_cre/seed_2021')
GDRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
print(f"✓ Đã kết nối Google Drive! Thư mục sao lưu: {GDRIVE_BACKUP_DIR}")

## 3. Clone Repository research-code-lab & Cài Đặt Môi Trường

In [ ]:
from pathlib import Path

WORKDIR = Path("/content")
%cd {WORKDIR}

# Clone repo chính thức trên nhánh feat/b5-wave-smoke
!git clone -b feat/b5-wave-smoke https://github.com/vinhlq2512/research-code-lab.git

REPO_DIR = WORKDIR / "research-code-lab"
WAVE_DIR = REPO_DIR / "paper-reproductions/WAVE-CRE"
%cd {WAVE_DIR}

# Cài đặt dependencies chính xác cho WAVE-CRE
!pip install -q transformers gpytorch linear_operator python-dotenv pyyaml

# Khởi tạo thư mục artifact theo quy chuẩn Protocol v1
RUN_DIR = REPO_DIR / "runs/wave_cre/fewrel_author_config/seed_2021"
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Lưu thông tin môi trường để bảo đảm reproducibility
!pip freeze > {RUN_DIR}/env_info.txt
!nvidia-smi > {RUN_DIR}/gpu_info.txt
print(f"✓ Môi trường đã sẵn sàng! Thư mục runs: {RUN_DIR}")

## 4. Chạy Thực Nghiệm Smoke 1 Seed (Seed 2021) trên GPU

In [ ]:
import subprocess
import sys

%cd {WAVE_DIR}

LOG_FILE = RUN_DIR / "run.log"
print(f"🚀 Bắt đầu thực thi WAVE-CRE seed 2021... Log ghi tại: {LOG_FILE}")

cmd = (
    "python -u run.py "
    "--max_length 256 "
    "--dataname FewRel "
    "--encoder_epochs 30 "
    "--encoder_lr 2e-5 "
    "--prompt_pool_epochs 25 "
    "--prompt_pool_lr 1e-4 "
    "--classifier_epochs 250 "
    "--seed 2021 "
    "--bert_path bert-base-uncased "
    "--data_path datasets "
    "--prompt_length 8 "
    "--prompt_top_k 4 "
    "--batch_size 16 "
    "--prompt_pool_size 20 "
    "--replay_s_e_e 100 "
    "--replay_epochs 200 "
    "--classifier_lr 5e-5 "
    "--prompt_type only_prompt"
)

# Chạy và stream log trực tiếp ra cell output đồng thời ghi vào file run.log
with open(LOG_FILE, "w") as f:
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        sys.stdout.write(line)
        f.write(line)
    process.wait()

if process.returncode == 0:
    print("\n✅ Thực nghiệm hoàn thành thành công!")
else:
    print(f"\n❌ Lỗi thực thi! Mã thoát: {process.returncode}")

## 5. Trích Xuất & Chuẩn Hóa Metrics Sang Chuẩn Protocol v1

In [ ]:
PARSE_SCRIPT = REPO_DIR / "scripts/parse_wave_results.py"
RESULTS_DIR = WAVE_DIR / "results/2021_coda_prompt"
OUTPUT_METRICS = RUN_DIR / "metrics.json"

!python {PARSE_SCRIPT} {RESULTS_DIR} --out {OUTPUT_METRICS} --meta seed=2021 dataset=FewRel model=WAVE-CRE

import json
if OUTPUT_METRICS.exists():
    with open(OUTPUT_METRICS, "r") as f:
        m = json.load(f)
    print("\n--- KẾT QUẢ THỰC NGHIỆM CHÍNH THỨC B5 (SWAG PREDICTED POOL) ---")
    print(json.dumps(m["derived_swag_predicted_pool"], indent=2))
else:
    print("⚠️ Chưa tìm thấy metrics.json. Kiểm tra lại log chạy.")

## 6. Tự Động Sao Lưu Toàn Bộ Artifacts Sang Google Drive

In [ ]:
import shutil

if Path('/content/drive/MyDrive').exists():
    print(f"Bắt đầu sao lưu từ {RUN_DIR} sang {GDRIVE_BACKUP_DIR}...")
    shutil.copytree(RUN_DIR, GDRIVE_BACKUP_DIR, dirs_exist_ok=True)
    print(f"✅ Đã sao lưu toàn bộ artifacts thành công lên Google Drive tại:")
    print(f"   {GDRIVE_BACKUP_DIR}")
else:
    print("⚠️ Google Drive chưa được mount. Hãy download thủ công thư mục runs trên Colab.")